# 02. 실험용 서브셋 구성 및 검증

## 구성 목적

**공식 Split 유지 → 각 Split 1/3 추출 → 소형 객체·희소 사례 보존 → 원본과 분포 비교 → Test Set 고정**

- 48시간 내 반복 학습이 가능한 데이터 규모 확보
- Train·Validation·Test 간 데이터 이동 없는 공식 Split 유지
- 소형 UAV의 bbox 크기 분포 보존
- 배경·다중 객체 등 희소 사례 보존

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_ROOT = PROJECT_ROOT / "data" / "yolo"
SUBSET_RATIO = 1/3
SPLITS = ["train", "val", "test"]
SEED = 42
AREA_BINS = 5
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

print(SOURCE_ROOT.resolve())

C:\Users\PMS\Desktop\MS\project\small-drone-detection\data\yolo


## 1. 이미지 단위 정보 생성

YOLO 라벨의 `width × height`를 bbox 면적 비율로 사용

이미지 경로, 라벨 경로, bbox 면적 목록과 객체 수를 한 번에 저장

In [2]:
def load_records(split):
    image_dir = SOURCE_ROOT / "images" / split
    label_dir = SOURCE_ROOT / "labels" / split
    records = []

    for image in sorted(image_dir.iterdir()):
        if image.suffix.lower() not in IMAGE_EXTS:
            continue

        label = label_dir / f"{image.stem}.txt"
        areas = []
        if label.exists():
            for line in label.read_text(encoding="utf-8").splitlines():
                values = line.split()
                if len(values) >= 5:
                    areas.append(float(values[3])*float(values[4]))

        records.append({"image": image, "label": label, "areas": areas, "object_count": len(areas)})
    return records

## 2. 서브셋 선정 기준

| 대상 | 선정 방법 | 근거 |
|---|---|---|
| 단일 객체 | bbox 크기순 5등분 후 각 구간에서 균등 추출 | 소형·중형·대형 객체 분포 보존 |
| Train 배경·다중 객체 | 전체 포함 | 빈도가 낮은 학습 사례 보존 |
| Validation·Test 배경·다중 객체 | 원본 비율에 맞춰 추출 | 평가 분포 왜곡 방지 |

Seed 42를 통한 동일 서브셋 재현

In [3]:
def sample_single(records, target_size, rng):
    if target_size >= len(records):
        return list(records)

    records = sorted(records, key=lambda x: x["areas"][0])
    groups = np.array_split(records, AREA_BINS)
    base, extra = divmod(target_size, AREA_BINS)

    selected = []
    for i, group in enumerate(groups):
        count = base + int(i < extra)
        selected += rng.sample(list(group), count)
    return selected


def select_subset(records, split, target_size, rng):
    empty = [x for x in records if x["object_count"] == 0]
    single = [x for x in records if x["object_count"] == 1]
    multi = [x for x in records if x["object_count"] >= 2]

    if split == "train":
        fixed = empty + multi
        selected = fixed + sample_single(single, target_size-len(fixed), rng)
    else:
        empty_count = round(len(empty)/len(records)*target_size)
        multi_count = round(len(multi)/len(records)*target_size)
        single_count = target_size-empty_count-multi_count
        selected = sample_single(single, single_count, rng)
        selected += rng.sample(empty, empty_count) if empty_count else []
        selected += rng.sample(multi, multi_count) if multi_count else []

    return sorted(selected, key=lambda x: x["image"].name)

## 3. 각 Split의 1/3 추출

공식 Split별 실제 이미지 수를 기준으로 목표 수 자동 계산

In [4]:
original, subset = {}, {}

for index, split in enumerate(SPLITS):
    records = load_records(split)
    target_size = round(len(records)*SUBSET_RATIO)
    selected = select_subset(records, split, target_size, random.Random(SEED+index))

    original[split], subset[split] = records, selected
    print(f"{split:5} | 원본 {len(records):4}장 | 서브셋 {len(selected):4}장 | 비율 {len(selected)/len(records):.3f}")

train | 원본 5200장 | 서브셋 1733장 | 비율 0.333
val   | 원본 2600장 | 서브셋  867장 | 비율 0.333
test  | 원본 2200장 | 서브셋  733장 | 비율 0.333


## 4. 원본과 서브셋 대표성 비교

이미지 수, 객체 수, 배경·다중 객체, bbox 중앙값과 소형 객체 비율을 하나의 표로 비교

In [5]:
def get_stats(records):
    areas = np.array([area for record in records for area in record["areas"]])
    multi = sum(record["object_count"] >= 2 for record in records)

    return {
        "이미지수": len(records),
        "객체수": len(areas),
        "배경이미지": sum(record["object_count"] == 0 for record in records),
        "다중객체": multi,
        "다중객체비율(%)": multi/len(records)*100,
        "bbox중앙값(%)": np.median(areas)*100,
        "1%미만(%)": (areas < 0.01).mean()*100,
        "5%미만(%)": (areas < 0.05).mean()*100,
    }


rows = []
for split in SPLITS:
    rows.append({"Split": split, "구분": "원본", **get_stats(original[split])})
    rows.append({"Split": split, "구분": "서브셋", **get_stats(subset[split])})

comparison_df = pd.DataFrame(rows)
display(comparison_df.round(4))

,Split,구분,이미지수,객체수,배경이미지,다중객체,다중객체비율(%),bbox중앙값(%),1%미만(%),5%미만(%)
0,train,원본,5200,5243,3,29,0.5577,0.0472,88.3273,94.8312
1,train,서브셋,1733,1776,3,29,1.6734,0.0489,87.2748,95.1577
2,val,원본,2600,2621,0,13,0.5000,0.0459,88.5540,94.2388
3,val,서브셋,867,876,0,4,0.4614,0.0457,89.3836,94.6347
4,test,원본,2200,2245,0,33,1.5000,0.0911,75.8575,92.8285
5,test,서브셋,733,749,0,11,1.5007,0.0941,76.3685,93.0574


## 5. bbox 분포 범위 비교

중앙값뿐 아니라 25%·75% 지점과 최솟값·최댓값을 확인하여 분포 일부만 우연히 일치하는 문제 점검

In [6]:
def get_bbox_range(records):
    areas = np.array([area*100 for record in records for area in record["areas"]])
    return {
        "bbox수": len(areas),
        "최솟값(%)": areas.min(),
        "25%(%)": np.percentile(areas, 25),
        "중앙값(%)": np.median(areas),
        "75%(%)": np.percentile(areas, 75),
        "최댓값(%)": areas.max(),
    }


range_rows = []
for split in SPLITS:
    range_rows.append({"Split": split, "구분": "원본", **get_bbox_range(original[split])})
    range_rows.append({"Split": split, "구분": "서브셋", **get_bbox_range(subset[split])})

display(pd.DataFrame(range_rows).round(4))

,Split,구분,bbox수,최솟값(%),25%(%),중앙값(%),75%(%),최댓값(%)
0,train,원본,5243,0.0019,0.0254,0.0472,0.1221,70.1988
1,train,서브셋,1776,0.0019,0.0257,0.0489,0.1496,64.8267
2,val,원본,2621,0.0000,0.0247,0.0459,0.1285,68.3894
3,val,서브셋,876,0.0034,0.0245,0.0457,0.1244,68.3894
4,test,원본,2245,0.0031,0.0347,0.0911,0.9306,47.2108
5,test,서브셋,749,0.0041,0.0347,0.0941,0.9253,47.2108


### 비교 결과

| Split | 확인 결과 |
|---|---|
| Train | bbox 중앙값 0.0472% → 0.0489%, 1% 미만 88.33% → 87.27% |
| Validation | bbox 중앙값 0.0459% → 0.0457%, 1% 미만 88.55% → 89.38% |
| Test | bbox 중앙값 0.0911% → 0.0941%, 1% 미만 75.86% → 76.37% |

모든 Split에서 bbox 중앙값과 소형 객체 비율의 유사성 확인

Train 다중 객체 비율 0.56% → 1.67% 증가는 다중 객체 29장을 전부 보존한 의도적 변화

Validation·Test 다중 객체 비율은 원본과 거의 동일한 수준 유지

## 6. 최종 판단

| 검증 항목 | 결과 | 판단 |
|---|---|---|
| 이미지 수 | Train 1,733, Validation 867, Test 733 | 전체의 약 1/3 확보 |
| bbox 크기 분포 | 중앙값과 1%·5% 미만 비율 유사 | 소형 객체 특성 보존 |
| Train 희소 사례 | 배경 3장과 다중 객체 29장 전체 포함 | 학습 정보 손실 방지 |
| Validation·Test 희소 사례 | 원본 비율 유지 | 평가 분포 보존 |
| 재현성 | Seed 42 고정 | 동일 서브셋 재생성 가능 |
| 데이터 누수 방지 | 공식 Split 간 이동 없음 | 독립 평가 구조 유지 |

서브셋은 원본의 핵심 분포를 유지하면서 반복 실험이 가능한 규모로 축소된 것으로 판단

Test 서브셋은 이후 모든 Baseline·개선 모델에서 변경 없이 고정

### 한계

- Train의 희소 사례 전체 포함에 따른 다중 객체 비율 증가
- 공식 Split 내부의 유사 연속 프레임 존재 가능성
- 전체 데이터가 아닌 서브셋 평가에 따른 일반화 해석 제한

보고서에 구성 기준과 분포 비교 수치를 함께 제시하여 서브셋 사용 근거 명시